Extract

In [34]:
import pandas as pd
import re

In [35]:
#comando para instalar a biblioteca da openai -> pip install openai
#1º Instalar a biblioteca da openai
#2º Acesse a plataforma da openai e crie uma chave de api e depois substitua o texto pela sua chave
#3º Importar a api e atribuir a nossa chave de api
#OBS: não deixar a chave salva aqui

#Controle do uso de IA:
USE_IA_REAL = False  # False se usar mock, true se usar a IA
#OpenAI (só será usado se USE_IA_REAL = True)
from openai import OpenAI
client = OpenAI(api_key="CHAVE_AQUI")

In [36]:
#Os dados serão inseridos em um arquivo com extensão csv, preferi criar dados "sujos" para trabalhar mais a parte da transformação
import pandas as pd

dados = [
    [1, "joão silva", "35", "001", "12345-6", "1111 2222 3333 4444", ""],
    [2, "MARIA Souza", "40 anos", "1", "123456", "1111222233334444", None],
    [3, "carlos   pereira", "trinta", "002", "98765-4", "4444 3333 2222 1111", ""],
    [4, "Ana Clara", "", "003", "11111-1", "1234 5678", ""],
    [5, "Pedro", "28", "", "22222-2", "9999 8888 7777 6666", ""],
    [6, "joão silva", "35", "001", "12345-6", "1111 2222 3333 4444", ""],  # duplicado
]

colunas = [
    "id", 
    "nome", 
    "idade", 
    "agencia", 
    "conta", 
    "cartao", 
    "mensagem_marketing"
]

df = pd.DataFrame(dados, columns=colunas)

df.to_csv("clientes.csv", index=False, encoding="utf-8")

print("CSV criado com sucesso!")

CSV criado com sucesso!


In [37]:
df = pd.read_csv("clientes.csv")
df.head()

,id,nome,idade,agencia,conta,cartao,mensagem_marketing
0,1,joão silva,35,1.0,12345-6,1111 2222 3333 4444,NaN
1,2,MARIA Souza,40 anos,1.0,123456,1111222233334444,NaN
2,3,carlos pereira,trinta,2.0,98765-4,4444 3333 2222 1111,NaN
3,4,Ana Clara,NaN,3.0,11111-1,1234 5678,NaN
4,5,Pedro,28,NaN,22222-2,9999 8888 7777 6666,NaN


TRANSFORM

In [38]:
#entender o dataSet
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  6 non-null      int64  
 1   nome                6 non-null      object 
 2   idade               5 non-null      object 
 3   agencia             5 non-null      float64
 4   conta               6 non-null      object 
 5   cartao              6 non-null      object 
 6   mensagem_marketing  0 non-null      float64
dtypes: float64(2), int64(1), object(4)
memory usage: 468.0+ bytes


In [39]:
#Gera estatísticas descritivas
df.describe(include="all")

,id,nome,idade,agencia,conta,cartao,mensagem_marketing
count,6.000000,6,5,5.000000,6,6,0.0
unique,NaN,5,4,NaN,5,5,NaN
top,NaN,joão silva,35,NaN,12345-6,1111 2222 3333 4444,NaN
freq,NaN,2,2,NaN,2,2,NaN
mean,3.500000,NaN,NaN,1.600000,NaN,NaN,NaN
std,1.870829,NaN,NaN,0.894427,NaN,NaN,NaN
min,1.000000,NaN,NaN,1.000000,NaN,NaN,NaN
25%,2.250000,NaN,NaN,1.000000,NaN,NaN,NaN
50%,3.500000,NaN,NaN,1.000000,NaN,NaN,NaN
75%,4.750000,NaN,NaN,2.000000,NaN,NaN,NaN


In [40]:
#mostrar os dados faltante
df.isnull().sum()

id                    0
nome                  0
idade                 1
agencia               1
conta                 0
cartao                0
mensagem_marketing    6
dtype: int64

In [54]:
#padronização dos nomes
df["nome"] = df["nome"].str.strip() #remover espaços vazios
df["nome"] = df["nome"].str.title() #padronizar

df["nome"]

0          João Silva
1         Maria Souza
2    Carlos   Pereira
3           Ana Clara
4               Pedro
5          João Silva
Name: nome, dtype: object

In [42]:
#padronização das idades
def limpar_idade(valor):
    if pd.isnull(valor) or valor == "":
        return None
    
    #extrai o número se estiver mesclado com texto
    numero = re.findall(r"\d+",str(valor))

    if numero:
        return int(numero[0])
    
    return None


In [43]:
#definir dados faltantes como nulo para correção posterior
df["idade"] = df["idade"].apply(limpar_idade)
df["agencia"] = df["agencia"].replace("", None)
df["conta"] = df["conta"].replace("", None)
df["cartao"] = df["cartao"].replace("", None)

df

,id,nome,idade,agencia,conta,cartao,mensagem_marketing
0,1,João Silva,35.0,1.0,12345-6,1111 2222 3333 4444,NaN
1,2,Maria Souza,40.0,1.0,123456,1111222233334444,NaN
2,3,Carlos Pereira,NaN,2.0,98765-4,4444 3333 2222 1111,NaN
3,4,Ana Clara,NaN,3.0,11111-1,1234 5678,NaN
4,5,Pedro,28.0,NaN,22222-2,9999 8888 7777 6666,NaN
5,6,João Silva,35.0,1.0,12345-6,1111 2222 3333 4444,NaN


In [44]:
#remover duplicados
df = df.drop_duplicates()

In [45]:
#criação de coluna para verificação da qualidade de dados
def verificar_dados(row):
    problemas = []

    if pd.isnull(row["idade"]):
        problemas.append("idade")
    if pd.isnull(row["agencia"]):
        problemas.append("agencia")
    if pd.isnull(row["conta"]):
        problemas.append("conta")
    if pd.isnull(row["cartao"]):
        problemas.append("cartao")

    return ", ".join(problemas) if problemas else "ok"

df["status_dados"] = df.apply(verificar_dados, axis=1)

df

,id,nome,idade,agencia,conta,cartao,mensagem_marketing,status_dados
0,1,João Silva,35.0,1.0,12345-6,1111 2222 3333 4444,NaN,ok
1,2,Maria Souza,40.0,1.0,123456,1111222233334444,NaN,ok
2,3,Carlos Pereira,NaN,2.0,98765-4,4444 3333 2222 1111,NaN,idade
3,4,Ana Clara,NaN,3.0,11111-1,1234 5678,NaN,idade
4,5,Pedro,28.0,NaN,22222-2,9999 8888 7777 6666,NaN,agencia
5,6,João Silva,35.0,1.0,12345-6,1111 2222 3333 4444,NaN,ok


In [46]:
#criação de coluna para definição de ação
def definir_acao(status):
    return "marketing" if status == "ok" else "atualizacao"

df["tipo_mensagem"] = df["status_dados"].apply(definir_acao)

df

,id,nome,idade,agencia,conta,cartao,mensagem_marketing,status_dados,tipo_mensagem
0,1,João Silva,35.0,1.0,12345-6,1111 2222 3333 4444,NaN,ok,marketing
1,2,Maria Souza,40.0,1.0,123456,1111222233334444,NaN,ok,marketing
2,3,Carlos Pereira,NaN,2.0,98765-4,4444 3333 2222 1111,NaN,idade,atualizacao
3,4,Ana Clara,NaN,3.0,11111-1,1234 5678,NaN,idade,atualizacao
4,5,Pedro,28.0,NaN,22222-2,9999 8888 7777 6666,NaN,agencia,atualizacao
5,6,João Silva,35.0,1.0,12345-6,1111 2222 3333 4444,NaN,ok,marketing


In [47]:
#FUNÇÃO IA REAL
def gerar_mensagem_real(cliente):
    nome = cliente["nome"]
    idade = cliente["idade"]
    status = cliente["status_dados"]
    tipo = cliente["tipo_mensagem"]

    if tipo == "marketing":
        prompt = f"""
        Gere uma mensagem de marketing bancário personalizada

        Nome: {nome}
        Idade: {idade}

        Seja amigável, direto e persuasivo
        """
    else:
        prompt = f"""
        Gere uma mensagem pedindo atualização de dados.append

        Nome: {nome}
        Dados faltantes: {status}

        Seja educado, claro e profissional
        """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Você é um assistente de marketing bancário."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    ) 

    return response.choices[0].message.content

In [48]:
#FUNÇÃO MOCK (como uma IA fake)
def gerar_mensagem_mock(cliente):
    nome = cliente["nome"]
    idade = cliente["idade"]
    tipo = cliente["tipo_mensagem"]
    status = cliente["status_dados"]
    
    # 🎯 Mensagem de marketing
    if tipo == "marketing":
        
        if idade is None:
            return f"Olá {nome}, temos ofertas exclusivas esperando por você! Confira nossos produtos financeiros."
        
        elif idade < 30:
            return f"Olá {nome}, que tal começar a investir no seu futuro? Temos opções ideais para você!"
        
        elif idade < 50:
            return f"Olá {nome}, temos ótimas condições de crédito e investimento para sua fase da vida."
        
        else:
            return f"Olá {nome}, conheça nossos planos de aposentadoria e segurança financeira."
    
    # 🎯 Mensagem de atualização
    else:
        return f"Olá {nome}, precisamos que você atualize os seguintes dados: {status}. Isso ajuda a manter sua conta segura e atualizada."

In [49]:
def gerar_mensagem(cliente):
    if USE_IA_REAL:
        try:
            return gerar_mensagem_real(cliente)
        except Exception as e:
            print(f"Erro na API, usando mock: {e}")
            return gerar_mensagem_mock(cliente)
    else:
        return gerar_mensagem_mock(cliente)


In [50]:
#teste em poucos registros
df.loc[0, "mensagem_marketing"] = gerar_mensagem(df.loc[0])
df.loc[1, "mensagem_marketing"] = gerar_mensagem(df.loc[1])

df.head()

C:\Users\SabrinaRP\AppData\Local\Temp\ipykernel_8148\230853255.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Olá João Silva, temos ótimas condições de crédito e investimento para sua fase da vida.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[0, "mensagem_marketing"] = gerar_mensagem(df.loc[0])


,id,nome,idade,agencia,conta,cartao,mensagem_marketing,status_dados,tipo_mensagem
0,1,João Silva,35.0,1.0,12345-6,1111 2222 3333 4444,"Olá João Silva, temos ótimas condições de créd...",ok,marketing
1,2,Maria Souza,40.0,1.0,123456,1111222233334444,"Olá Maria Souza, temos ótimas condições de cré...",ok,marketing
2,3,Carlos Pereira,NaN,2.0,98765-4,4444 3333 2222 1111,NaN,idade,atualizacao
3,4,Ana Clara,NaN,3.0,11111-1,1234 5678,NaN,idade,atualizacao
4,5,Pedro,28.0,NaN,22222-2,9999 8888 7777 6666,NaN,agencia,atualizacao


In [51]:
#Aplicação do df inteiro
df["mensagem_marketing"] = df.apply(gerar_mensagem, axis=1)

In [52]:
df[["nome", "tipo_mensagem", "status_dados", "mensagem_marketing"]]

,nome,tipo_mensagem,status_dados,mensagem_marketing
0,João Silva,marketing,ok,"Olá João Silva, temos ótimas condições de créd..."
1,Maria Souza,marketing,ok,"Olá Maria Souza, temos ótimas condições de cré..."
2,Carlos Pereira,atualizacao,idade,"Olá Carlos Pereira, precisamos que você atua..."
3,Ana Clara,atualizacao,idade,"Olá Ana Clara, precisamos que você atualize os..."
4,Pedro,atualizacao,agencia,"Olá Pedro, precisamos que você atualize os seg..."
5,João Silva,marketing,ok,"Olá João Silva, temos ótimas condições de créd..."


LOAD

In [53]:
#Salvando o resultado final
df.to_csv("clientes_final.csv", index=False)